In [1]:
import numpy as np 
import pandas as pd 
import seaborn as sns
import matplotlib.pyplot as plt 
import warnings 
warnings.filterwarnings('ignore')

In [2]:
df=pd.read_csv('covid_toy.csv')

In [3]:
df.sample(5)

,age,gender,fever,cough,city,has_covid
83,17,Female,104.0,Mild,Kolkata,No
5,84,Female,NaN,Mild,Bangalore,Yes
12,25,Female,99.0,Strong,Kolkata,No
97,20,Female,101.0,Mild,Bangalore,No
36,38,Female,101.0,Mild,Bangalore,No


In [4]:
df.isnull().sum()

age           0
gender        0
fever        10
cough         0
city          0
has_covid     0
dtype: int64

In [5]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.model_selection import train_test_split

In [6]:
X_train,X_test,y_train,y_test=train_test_split(df.drop(columns=['has_covid'],axis=1),
                                               df['has_covid'],
                                              test_size=0.2,
                                               random_state=42)
X_train.shape,

((80, 5),)

## AAm Gindgi|

In [15]:
df['fever'].value_counts()

fever
101.0    17
98.0     17
104.0    14
100.0    13
99.0     10
102.0    10
103.0     9
Name: count, dtype: int64

In [8]:
# add simpleimputer to fever col 
si=SimpleImputer() 
X_train_fever=si.fit_transform(X_train[['fever']])
# also for test data 
X_test_fever=si.transform(X_test[['fever']])
X_train_fever.shape

(80, 1)

In [17]:
df['cough'].value_counts()

cough
Mild      62
Strong    38
Name: count, dtype: int64

In [10]:
# OrdinalEncoder for cough
oe=OrdinalEncoder(categories=[['Mild','Strong']])
X_train_cough=oe.fit_transform(X_train[['cough']])
X_test_cough=oe.transform(X_test[['cough']])
X_train_cough.shape

(80, 1)

In [18]:
df['gender'].value_counts()

gender
Female    59
Male      41
Name: count, dtype: int64

In [19]:
# Now apply in OneHotEncoder in Gender and city 
oh=OneHotEncoder(drop='first',sparse_output=False)
X_train_gender_city=oh.fit_transform(X_train[['gender','city']])
X_test_gender_city=oh.transform(X_test[['gender','city']])
X_train_gender_city.shape

(80, 4)

In [20]:
# Extraction data 
X_train_age=X_train.drop(columns=['gender','cough','fever','city']).values
# Also for test data
X_test_age=X_test.drop(columns=['gender','cough','fever','city']).values
X_train_age.shape

(80, 1)

In [27]:
X_train_transformer=np.concatenate((X_train_age,X_train_fever,X_train_cough,X_train_gender_city),axis=1)
X_test_transformer=np.concatenate((X_test_age,X_test_cough,X_test_fever,X_test_gender_city),axis=1)
X_train_transformer

array([[ 81., 101.,   0.,   0.,   0.,   0.,   1.],
       [  5., 100.,   0.,   0.,   0.,   1.,   0.],
       [ 19., 100.,   0.,   0.,   0.,   1.,   0.],
       [ 27., 100.,   0.,   1.,   1.,   0.,   0.],
       [ 73., 103.,   0.,   0.,   1.,   0.,   0.],
       [ 70., 103.,   1.,   1.,   0.,   1.,   0.],
       [ 49., 102.,   0.,   0.,   1.,   0.,   0.],
       [ 51., 101.,   1.,   0.,   0.,   1.,   0.],
       [ 64., 101.,   0.,   0.,   1.,   0.,   0.],
       [ 83., 101.,   0.,   0.,   0.,   1.,   0.],
       [ 65.,  98.,   0.,   0.,   0.,   0.,   1.],
       [ 18., 104.,   0.,   0.,   0.,   0.,   0.],
       [ 16., 103.,   0.,   0.,   0.,   0.,   0.],
       [ 16., 104.,   0.,   1.,   0.,   1.,   0.],
       [ 27., 100.,   0.,   1.,   0.,   1.,   0.],
       [ 84., 101.,   0.,   0.,   0.,   0.,   0.],
       [ 51., 104.,   0.,   1.,   0.,   1.,   0.],
       [ 69., 102.,   0.,   0.,   0.,   0.,   0.],
       [ 82., 102.,   1.,   0.,   0.,   0.,   0.],
       [ 69., 103.,   0.,   0.,

## Mentos Gindgi

In [25]:
from sklearn.compose import ColumnTransformer
transformer=ColumnTransformer(transformers=[
    ('tnf1',SimpleImputer(),['fever']),
    ('tnf2',OrdinalEncoder(categories=[['Mild','Strong']]),['cough']),
    ('tnf3',OneHotEncoder(drop='first',sparse_output=False),['gender','city'])],
                              remainder='passthrough')

In [26]:
transformer.fit_transform(X_train)

array([[101.,   0.,   0.,   0.,   0.,   1.,  81.],
       [100.,   0.,   0.,   0.,   1.,   0.,   5.],
       [100.,   0.,   0.,   0.,   1.,   0.,  19.],
       [100.,   0.,   1.,   1.,   0.,   0.,  27.],
       [103.,   0.,   0.,   1.,   0.,   0.,  73.],
       [103.,   1.,   1.,   0.,   1.,   0.,  70.],
       [102.,   0.,   0.,   1.,   0.,   0.,  49.],
       [101.,   1.,   0.,   0.,   1.,   0.,  51.],
       [101.,   0.,   0.,   1.,   0.,   0.,  64.],
       [101.,   0.,   0.,   0.,   1.,   0.,  83.],
       [ 98.,   0.,   0.,   0.,   0.,   1.,  65.],
       [104.,   0.,   0.,   0.,   0.,   0.,  18.],
       [103.,   0.,   0.,   0.,   0.,   0.,  16.],
       [104.,   0.,   1.,   0.,   1.,   0.,  16.],
       [100.,   0.,   1.,   0.,   1.,   0.,  27.],
       [101.,   0.,   0.,   0.,   0.,   0.,  84.],
       [104.,   0.,   1.,   0.,   1.,   0.,  51.],
       [102.,   0.,   0.,   0.,   0.,   0.,  69.],
       [102.,   1.,   0.,   0.,   0.,   0.,  82.],
       [103.,   0.,   0.,   0.,

In [28]:
transformer.transform(X_test)

array([[104.,   0.,   0.,   0.,   1.,   0.,  17.],
       [ 98.,   0.,   1.,   1.,   0.,   0.,  83.],
       [101.,   1.,   0.,   1.,   0.,   0.,  68.],
       [ 99.,   0.,   1.,   0.,   0.,   0.,  72.],
       [102.,   1.,   1.,   1.,   0.,   0.,  20.],
       [103.,   0.,   0.,   0.,   1.,   0.,  50.],
       [ 98.,   1.,   0.,   0.,   1.,   0.,  71.],
       [ 99.,   0.,   0.,   0.,   0.,   1.,  14.],
       [101.,   0.,   0.,   1.,   0.,   0.,  75.],
       [103.,   0.,   1.,   0.,   1.,   0.,  60.],
       [ 98.,   0.,   0.,   0.,   0.,   0.,  64.],
       [101.,   0.,   1.,   1.,   0.,   0.,  15.],
       [ 98.,   1.,   1.,   0.,   1.,   0.,  34.],
       [ 98.,   0.,   0.,   0.,   1.,   0.,  26.],
       [ 99.,   1.,   0.,   1.,   0.,   0.,  59.],
       [101.,   0.,   0.,   0.,   0.,   1.,  65.],
       [100.,   0.,   1.,   0.,   0.,   0.,  80.],
       [101.,   0.,   0.,   0.,   1.,   0.,   8.],
       [ 99.,   1.,   0.,   0.,   1.,   0.,  25.],
       [103.,   0.,   1.,   0.,